# Re-run NSGA-II in place

Surgical re-run notebook for the NSGA-II rows of two existing experiment
tables in `artifacts/results/`:

* **§12 benchmark sweep** -- `benchmark_sweep.csv` (one row per `(city,
  method)`) and `benchmark_<city>_routes.pt` (route payload for the route
  figures). Updated per city: the existing `(city, "NSGA-II")` row is
  dropped and replaced with a fresh run; the `RunResult` for NSGA-II inside
  `benchmark_<city>_routes.pt` is likewise swapped. Every other method stays
  untouched, so the rest of the sweep does not need to be re-run.
* **§9c Mumford0 standalone** -- `nsgaii_comparison.csv` (Initial LC +
  NSGA-II row + delta-vs-initial columns). Optional, gated by
  `RERUN_NSGAII_MUMFORD0_STANDALONE` below; this run uses the §9c route
  contract (`N_ROUTES=10`, `MAX_ROUTE_LEN=12`, seed = `lc_base_routes`),
  which differs from the §12 Mumford0 contract (`12 routes / max_len=15`,
  seed = NX-heuristic).

The new NSGA-II runs use the seeded variant added to
`connectpt.routes_generator.nsgaii`: the NX-heuristic init (§12) or the LC-base
routes (§9c) are injected into the initial population as one explicit member,
the remaining `pop_size - 1` slots are filled via `cfg.init_mode` (default
`husselmann`).

## 1. Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# eval_lib must be importable -- the notebook's working directory holds it.
import os as _os, sys as _sys
_NB_DIR = _os.path.abspath(".")
if _NB_DIR not in _sys.path:
    _sys.path.insert(0, _NB_DIR)
import eval_lib
from eval_lib import *
from eval_lib import _run_baseline  # underscore: skipped by `import *`

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print("eval_lib OK; RESULTS_DIR =", RESULTS_DIR.relative_to(ROOT_DIR))

## 2. Configuration

Pick which benchmark cities to re-run and which result files to update.
Defaults: re-run every benchmark city for §12; skip §9c. Mumford2/3 NSGA-II
are slow -- subset `RERUN_CITIES` if needed.

In [ ]:
# Which benchmark cities to re-run NSGA-II on. Subset for partial reruns.
RERUN_CITIES = [spec["city"] for spec in BENCHMARK_SPECS]

# Update the existing benchmark_sweep.csv (drop old NSGA-II rows for the chosen
# cities, append new ones; every other method row stays as is).
UPDATE_BENCHMARK_SWEEP_CSV = True

# Update the per-city benchmark_<city>_routes.pt (swap NSGA-II RunResult inside
# each saved payload; every other run stays as is).
UPDATE_BENCHMARK_PT = True

# Re-render benchmark route figures from the updated .pt files.
REDRAW_BENCHMARK_FIGURES = True

# §9c Mumford0 standalone NSGA-II uses a different route contract (N_ROUTES=10,
# max_len=12, seed = lc_base_routes). Set True to also rerun that section and
# refresh nsgaii_comparison.csv.
RERUN_NSGAII_MUMFORD0_STANDALONE = False

print(f"Cities to re-run NSGA-II on: {RERUN_CITIES}")
print(f"Update benchmark_sweep.csv: {UPDATE_BENCHMARK_SWEEP_CSV}")
print(f"Update benchmark_<city>_routes.pt: {UPDATE_BENCHMARK_PT}")
print(f"Redraw benchmark figures: {REDRAW_BENCHMARK_FIGURES}")
print(f"Rerun §9c Mumford0 standalone: {RERUN_NSGAII_MUMFORD0_STANDALONE}")

## 3. Re-run NSGA-II per benchmark city

For each city in `RERUN_CITIES`:

1. Build the same NX-heuristic init routes the rest of the §12 sweep uses
   via `load_benchmark_graph(spec)`.
2. Run NSGA-II with that init as the seed (`init_routes=`).
3. Reduce the Pareto front to the lowest weighted-sum member under
   `(DEMAND_TIME_WEIGHT, ROUTE_TIME_WEIGHT)`.
4. Evaluate the chosen routes via the shared `_run_baseline` path so the
   metric columns line up with the existing `benchmark_sweep.csv`.

Captures `(metrics, routes)` per city into `new_nsgaii_runs` for the
update steps below.

In [ ]:
import gc

NSGAII_METHOD_LABEL = "NSGA-II"
new_nsgaii_runs = {}  # city -> {"metrics": ..., "routes": ..., "row": ...}

_specs_by_city = {spec["city"]: spec for spec in BENCHMARK_SPECS}
for city in RERUN_CITIES:
    if city not in _specs_by_city:
        print(f"[skip] {city}: not in BENCHMARK_SPECS")
        continue
    spec = _specs_by_city[city]
    n_routes = spec["n_routes"]
    min_len = spec["min_route_len"]
    max_len = spec["max_route_len"]
    print(f"=== {city}: n_routes={n_routes} min={min_len} max={max_len} ===")
    try:
        tensors, init_routes = load_benchmark_graph(spec)
    except Exception as exc:
        print(f"  [{city}] could not build NX-heuristic init: {exc}")
        continue

    try:
        _, output = run_nsgaii(
            build_nsgaii_cfg(f"{city}_nsgaii_rerun", n_routes, min_len, max_len),
            tensors=tensors, init_routes=init_routes,
            run_name_scope=f"{city}_rerun_")
        best = reduce_pareto_front(
            output, DEMAND_TIME_WEIGHT, ROUTE_TIME_WEIGHT)
        routes = best["routes"]
        if routes.ndim == 2:
            routes = routes[None]
        eval_name, metrics, _, eval_routes = _run_baseline(
            None,
            build_sa_cfg(f"{city}_nsgaii_rerun_eval", n_routes, min_len, max_len),
            routes, f"{city}_nsgaii_rerun_eval_", {}, tensors=tensors)
        row = summarize_benchmark_run(city, NSGAII_METHOD_LABEL, metrics)
        new_nsgaii_runs[city] = {
            "metrics": metrics,
            "routes": as_route_tensor(eval_routes),
            "row": row,
        }
        print(f"  [{city}] NSGA-II rerun cost={metrics.get('cost', float('nan')):.4f}")
    except Exception as exc:
        print(f"  [{city}] NSGA-II rerun FAILED: {exc!r}")
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\nFinished {len(new_nsgaii_runs)} / {len(RERUN_CITIES)} city reruns.")

## 4. Update `benchmark_sweep.csv`

Load the existing CSV, drop every `(city, "NSGA-II")` row whose city we
just re-ran, append the fresh rows, save back. Every other method row stays
exactly as it was -- if the file did not exist (first run on this machine),
we create it with just the NSGA-II rows.

In [ ]:
if UPDATE_BENCHMARK_SWEEP_CSV and new_nsgaii_runs:
    csv_path = RESULTS_DIR / "benchmark_sweep.csv"
    if csv_path.exists():
        existing_df = pd.read_csv(csv_path)
        before = len(existing_df)
        # Drop the (city, NSGA-II) rows we are about to replace.
        mask_drop = (existing_df["method"] == NSGAII_METHOD_LABEL) & \
                    existing_df["benchmark"].isin(list(new_nsgaii_runs.keys()))
        kept_df = existing_df.loc[~mask_drop].copy()
        dropped = before - len(kept_df)
        print(f"benchmark_sweep.csv: dropped {dropped} stale NSGA-II rows; "
              f"keeping {len(kept_df)} rows for other methods.")
    else:
        kept_df = pd.DataFrame()
        print("benchmark_sweep.csv: did not exist; creating fresh.")

    new_rows = [info["row"] for info in new_nsgaii_runs.values()]
    new_df = pd.concat([kept_df, pd.DataFrame(new_rows)], ignore_index=True)
    save_table(new_df, "benchmark_sweep")
    display(new_df.loc[new_df["method"] == NSGAII_METHOD_LABEL].round(3))
else:
    print("Skipped benchmark_sweep.csv update.")

## 5. Update `benchmark_<city>_routes.pt`

For every re-run city: reload its route payload, find the existing
`RunResult` with `label == "NSGA-II"`, swap its `routes` / `metrics` for the
fresh run, and save the payload back via `save_route_results`. If no NSGA-II
run was in the payload (e.g. the original sweep skipped it), we append a new
`RunResult` instead. Coords + street_adj are preserved from the loaded
payload so the figure is reproducible.

In [ ]:
from eval_lib import RunResult, load_route_results, save_route_results

if UPDATE_BENCHMARK_PT and new_nsgaii_runs:
    for city, info in new_nsgaii_runs.items():
        payload_name = f"benchmark_{city}"
        try:
            results, coords, street_adj = load_route_results(payload_name)
        except FileNotFoundError:
            print(f"  [{city}] {payload_name}_routes.pt missing -- skipping "
                  f"(re-run the full §12 sweep first).")
            continue

        replaced = False
        for i, r in enumerate(results):
            if r.label == NSGAII_METHOD_LABEL:
                results[i] = RunResult(
                    label=NSGAII_METHOD_LABEL,
                    kind=getattr(r, "kind", "bco"),
                    dataset=getattr(r, "dataset", city),
                    routes=info["routes"],
                    metrics=info["metrics"],
                )
                replaced = True
                break
        if not replaced:
            results.append(RunResult(
                label=NSGAII_METHOD_LABEL, kind="bco", dataset=city,
                routes=info["routes"], metrics=info["metrics"]))

        save_route_results(results, payload_name,
                           coords=coords, street_adj=street_adj)
        print(f"  [{city}] {'replaced' if replaced else 'appended'} "
              f"NSGA-II run in {payload_name}_routes.pt")
else:
    print("Skipped benchmark_<city>_routes.pt update.")

## 6. Re-render benchmark route figures

Pull the updated `.pt` payloads back through `render_route_comparison_figure`
so the saved-to-disk visualisations reflect the new NSGA-II routes. Cell 0
is the NX-heuristic initial, cells 1.. are every method (including the
just-replaced NSGA-II).

In [ ]:
if REDRAW_BENCHMARK_FIGURES and new_nsgaii_runs:
    for city in new_nsgaii_runs:
        payload_name = f"benchmark_{city}"
        try:
            results, coords, street_adj = load_route_results(payload_name)
        except FileNotFoundError:
            print(f"  [{city}] {payload_name}_routes.pt missing -- skipping redraw.")
            continue
        if not results:
            print(f"  [{city}] empty payload -- skipping redraw.")
            continue
        reference = next((r for r in results if r.kind == "initial"), results[0])
        cases = [r for r in results if r is not reference]
        render_route_comparison_figure(
            reference, cases, coords, street_adj,
            title=f"{city} benchmark -- route comparison (NSGA-II refreshed)",
            ncols=4, palette="tab10", with_overlap_curves=False)
        plt.show()
else:
    print("Skipped figure redraw.")

## 7. (Optional) §9c Mumford0 standalone NSGA-II

If `RERUN_NSGAII_MUMFORD0_STANDALONE = True`, re-run the smaller §9c
NSGA-II comparison (`N_ROUTES=10`, `MAX_ROUTE_LEN=12`, seed = LC-base
routes from `run_lc_base()`) and refresh `nsgaii_comparison.csv` and the
Pareto-front scatter.

This is independent of the §12 benchmark rerun above -- the route contract
and the seed network are different, so the two NSGA-II runs are not
comparable point-for-point.

In [ ]:
if RERUN_NSGAII_MUMFORD0_STANDALONE:
    # Build the §9c LC-base routes (same call the main notebook makes in §5).
    base_result = run_lc_base()
    lc_base_metrics = base_result["metrics"]
    lc_base_routes = base_result["routes"]
    print(f"§9c LC-base: cost={lc_base_metrics.get('cost', float('nan')):.4f}, "
          f"routes shape={tuple(lc_base_routes.shape)}")

    nsgaii_run_name, nsgaii_output = run_nsgaii(
        build_nsgaii_cfg("nsgaii_mumford0_rerun",
                         N_ROUTES, MIN_ROUTE_LEN, MAX_ROUTE_LEN),
        init_routes=lc_base_routes)
    nsgaii_front = nsgaii_output["pareto_pop"]
    print(f"[{nsgaii_run_name}] Pareto front members: {len(nsgaii_front)}")

    nsgaii_best = reduce_pareto_front(
        nsgaii_output, DEMAND_TIME_WEIGHT, ROUTE_TIME_WEIGHT)
    nsgaii_routes = nsgaii_best["routes"]
    if nsgaii_routes.ndim == 2:
        nsgaii_routes = nsgaii_routes[None]
    nsgaii_eval_name, nsgaii_metrics, _, nsgaii_routes = _run_baseline(
        None,
        build_sa_cfg("nsgaii_mumford0_rerun_eval",
                     N_ROUTES, MIN_ROUTE_LEN, MAX_ROUTE_LEN),
        nsgaii_routes, "nsgaii_rerun_eval_", {})

    # Pareto-front scatter, same shape as the original §9c figure.
    _front_costs = np.array([m["cost"] for m in nsgaii_front], dtype=float)
    fig, ax = plt.subplots(figsize=(5.5, 4))
    ax.scatter(_front_costs[:, 0], _front_costs[:, 1], c="tab:blue",
               label="Pareto front")
    ax.scatter([float(nsgaii_best["cost"][0])],
               [float(nsgaii_best["cost"][1])],
               c="tab:red", s=90, marker="*", zorder=3,
               label="weighted-sum pick")
    ax.set_xlabel("mean demand time (objective 1)")
    ax.set_ylabel("total route time (objective 2)")
    ax.set_title("NSGA-II Pareto front on Mumford0 (rerun, seeded with LC-base)")
    ax.legend()
    plt.show()

    nsgaii_rows = [
        {**summarize_run("Initial LC", lc_base_metrics, lc_base_routes),
         "optimizer": "Initial LC"},
        {**summarize_run("NSGA-II (weighted-sum pick)",
                         nsgaii_metrics, nsgaii_routes),
         "optimizer": "NSGA-II"},
    ]
    nsgaii_comparison_df = pd.DataFrame(nsgaii_rows)
    _init_row = nsgaii_comparison_df.loc[
        nsgaii_comparison_df["method"] == "Initial LC"].iloc[0]
    nsgaii_comparison_df["delta_cost_vs_initial_lc"] = (
        nsgaii_comparison_df["cost"] - _init_row["cost"])
    nsgaii_comparison_df["delta_ATT_vs_initial_lc"] = (
        nsgaii_comparison_df["ATT"] - _init_row["ATT"])
    nsgaii_comparison_df["delta_RTT_vs_initial_lc"] = (
        nsgaii_comparison_df["RTT"] - _init_row["RTT"])

    from eval_lib import plots as _plots
    nsgaii_comparison_df = nsgaii_comparison_df[_plots.filter_component_columns([
        "method", "optimizer", "cost",
        "cost_demand_term", "cost_route_term",
        "cost_connectivity_term", "cost_penalty_term",
        "ATT", "RTT", "median_connectivity",
        "$d_{un}$", "# disconnected node pairs",
        "# stops out of bounds", "# routes",
        "delta_cost_vs_initial_lc", "delta_ATT_vs_initial_lc",
        "delta_RTT_vs_initial_lc",
    ], ENABLED_COST_COMPONENTS)]

    save_table(nsgaii_comparison_df, "nsgaii_comparison")
    display(nsgaii_comparison_df.round(3))
else:
    print("Skipped §9c Mumford0 standalone NSGA-II rerun "
          "(set RERUN_NSGAII_MUMFORD0_STANDALONE=True to include it).")

## What got updated

* `artifacts/results/benchmark_sweep.csv` -- stale `(city, "NSGA-II")` rows
  swapped, every other method untouched.
* `artifacts/results/benchmark_<city>_routes.pt` -- NSGA-II `RunResult`
  swapped per city; figures redrawn from the refreshed payloads.
* `artifacts/results/nsgaii_comparison.csv` -- only if the §9c block above
  ran (it is gated by `RERUN_NSGAII_MUMFORD0_STANDALONE`).

Files this notebook never touches: `worse_accept_*`, `nx_dataset_*`,
`mumford0_lc_*`, `macsa_*`, `objective_weight_*`, `train_case_*`,
`baseline_optimizers_*`. Those experiments are unaffected.